### Game Genie Room Setup

Creates a Genie room for Level 1 of Casper's Kitchen Rescue.
The room is configured with the investigation views as trusted assets
and includes table-level instructions that guide discovery without
giving away answers.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create Genie Room via SDK

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get a SQL warehouse for the Genie room
warehouses = list(w.warehouses.list())
warehouse_id = None
for wh in warehouses:
    if wh.name and CATALOG in wh.name:
        warehouse_id = wh.id
        break

# Fallback: use any available serverless warehouse
if not warehouse_id:
    for wh in warehouses:
        if wh.enable_serverless_compute or 'serverless' in (wh.name or '').lower():
            warehouse_id = wh.id
            break

# Last resort: use any warehouse
if not warehouse_id and warehouses:
    warehouse_id = warehouses[0].id

if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create one first.")

print(f"Using warehouse_id={warehouse_id}")

In [ ]:
table_identifiers = [
    f"{CATALOG}.game.investigation_revenue_hourly",
    f"{CATALOG}.game.investigation_brand_sales",
    f"{CATALOG}.lakeflow.gold_order_header",
    f"{CATALOG}.simulator.locations",
    f"{CATALOG}.simulator.brands",
]

description = """You are helping investigate a revenue anomaly at Casper's Kitchens, a ghost kitchen chain.

Management has reported that revenue is down at one of the locations, particularly during certain hours.
One brand seems to be hit harder than others.

Your job: use natural language questions to explore the data and find:
1. Which location has the revenue drop
2. What time of day the drop is most visible
3. Which brand is most affected

Tables available:
- investigation_revenue_hourly: hourly revenue and orders by location
- investigation_brand_sales: daily brand-level revenue
- gold_order_header: per-order summary
- locations: location names and details
- brands: brand names

Start by asking about total revenue by location, then drill down into hourly patterns."""

print("Genie room configuration ready")
print(f"Tables: {table_identifiers}")

In [ ]:
import json, requests

genie_url = None
host = w.config.host.rstrip("/")
room_title = f"Casper's Kitchen Investigation ({CATALOG})"

# Get auth token (works in notebook context)
try:
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
except Exception:
    token = w.config.token
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print(f"Host: {host}")
print(f"Auth token obtained: {'yes' if token else 'NO'}")

# 1. Try SDK first (most reliable in notebook context)
print("\n--- Attempt 1: SDK ---")
try:
    genie_space = w.genie.create_space(
        title=room_title,
        description=description,
        warehouse_id=warehouse_id,
        table_identifiers=table_identifiers,
    )
    space_id = genie_space.space_id
    genie_url = f"{host}/genie/rooms/{space_id}"
    print(f"\u2705 Created Genie room via SDK: {genie_url}")
except Exception as e:
    print(f"SDK failed: {type(e).__name__}: {e}")

# 2. REST API fallback
if not genie_url:
    print("\n--- Attempt 2: REST API ---")
    # Check for existing room first
    try:
        list_resp = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
        print(f"  List spaces: {list_resp.status_code}")
        if list_resp.ok:
            for space in list_resp.json().get("spaces", []):
                if space.get("title") == room_title:
                    space_id = space["space_id"]
                    genie_url = f"{host}/genie/rooms/{space_id}"
                    print(f"♻️ Found existing Genie room: {genie_url}")
                    break
    except Exception as e:
        print(f"  List failed: {e}")

    if not genie_url:
        payload = {
            "title": room_title,
            "description": description,
            "warehouse_id": warehouse_id,
            "table_identifiers": table_identifiers,
        }
        try:
            resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload)
            print(f"  Create status: {resp.status_code}")
            print(f"  Create body: {resp.text[:500]}")
            if resp.ok:
                data = resp.json()
                space_id = data.get("space_id") or data.get("id")
                if space_id:
                    genie_url = f"{host}/genie/rooms/{space_id}"
                    print(f"\u2705 Created Genie room: {genie_url}")
                else:
                    print(f"\u26a0\ufe0f Response has no space_id: {data}")
        except Exception as e:
            print(f"  Create failed: {e}")

if not genie_url:
    print("\n\u274c Could not create Genie room.")
    print("Create one manually in the Databricks UI with these tables:")
    for t in table_identifiers:
        print(f"  - {t}")
    print(f"\nThen run this SQL:")
    print(f"  MERGE INTO {CATALOG}.game.config AS t")
    print(f"  USING (SELECT 'genie_room_url' AS config_key, '<URL>' AS config_value) AS s")
    print(f"  ON t.config_key = s.config_key")
    print(f"  WHEN MATCHED THEN UPDATE SET config_value = s.config_value")
    print(f"  WHEN NOT MATCHED THEN INSERT VALUES (s.config_key, s.config_value);")
else:
    print(f"\n\u2705 Genie room ready: {genie_url}")

In [ ]:
# Store the Genie room URL in the game config table for the quest controller app
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.config (
  config_key STRING,
  config_value STRING
)
""")

if genie_url:
    spark.sql(f"""
    MERGE INTO {CATALOG}.game.config AS target
    USING (SELECT 'genie_room_url' AS config_key, '{genie_url}' AS config_value) AS source
    ON target.config_key = source.config_key
    WHEN MATCHED THEN UPDATE SET config_value = source.config_value
    WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
    """)
    print(f"\u2705 Stored Genie room URL in game config")
else:
    print("\u26a0\ufe0f No Genie URL to store. Set it manually in game.config after creating the room.")